# 3.5 - Ensemble & Evaluation: Model Stacking & Backtesting

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

**Fase final del pipeline de modelado:**

1. **Ensemble (Stacking):** Combinar predicciones de múltiples modelos
2. **Walk-Forward Validation:** Backtesting realista con ventana deslizante
3. **Trading Simulation:** Evaluar en contexto de portafolio
4. **Model Selection:** Identificar mejor modelo por commodity

**Stacking:**
- Meta-modelo que aprende a ponderar predicciones de modelos base
- Captura fortalezas complementarias (tree + time series + linear)
- Típicamente supera a modelos individuales

**Walk-Forward:**
- Simula predicción en tiempo real (no mira hacia el futuro)
- Entrena en ventana deslizante, predice siguiente período
- Más realista que single train/test split

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm
from time import perf_counter
import warnings
warnings.filterwarnings('ignore')

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Definir commodities target
TARGET_COMMODITIES = ['Corn', 'Soybeans', 'Wheat']

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Target commodities: {', '.join(TARGET_COMMODITIES)}")

## 1. Cargar Datos y Modelos Pre-entrenados

In [ ]:
# Cargar dataset con features seleccionadas
input_file = PROCESSED_DIR / 'features_selected_modeling.csv'

if not input_file.exists():
    raise FileNotFoundError(f"No se encontró {input_file}. Ejecuta notebook 3.1 primero.")

df = pd.read_csv(input_file, parse_dates=['date'])

print(f"✓ Dataset cargado: {input_file.name}")
print(f"  Dimensiones: {df.shape}")

# Separar features y targets
target_cols = [f'{c}_target_t7' for c in TARGET_COMMODITIES]
feature_cols = [c for c in df.columns if c not in ['date'] + target_cols]

# Split temporal
split_date = '2023-01-01'
train_idx = df['date'] < split_date
test_idx = df['date'] >= split_date

X_train = df.loc[train_idx, feature_cols]
X_test = df.loc[test_idx, feature_cols]
y_train = df.loc[train_idx, target_cols]
y_test = df.loc[test_idx, target_cols]

print(f"\nTrain/Test split:")
print(f"  Train: {X_train.shape[0]:,} obs")
print(f"  Test: {X_test.shape[0]:,} obs")
print(f"  Features: {len(feature_cols)}")

In [ ]:
# Cargar modelos pre-entrenados
models_dir = BASE_DIR / 'models'

# Baseline models
baseline_file = models_dir / 'baseline_models.pkl'
if baseline_file.exists():
    with open(baseline_file, 'rb') as f:
        baseline_models = pickle.load(f)
    print(f"✓ Baseline models cargados: {list(baseline_models.keys())}")
else:
    baseline_models = None
    print("⚠ No se encontraron baseline models")

# Tree models
tree_file = models_dir / 'tree_models.pkl'
if tree_file.exists():
    with open(tree_file, 'rb') as f:
        tree_models = pickle.load(f)
    print(f"✓ Tree models cargados: {list(tree_models.keys())}")
else:
    tree_models = None
    print("⚠ No se encontraron tree models")

# Verificar si hay modelos cargados
if baseline_models is None and tree_models is None:
    raise FileNotFoundError("No se encontraron modelos pre-entrenados. Ejecuta notebooks 3.2 y 3.3 primero.")

---

## 2. Ensemble: Stacking de Modelos

**Stacking workflow:**
1. Generar predicciones de modelos base en train/test
2. Usar predicciones como features para meta-modelo
3. Meta-modelo aprende pesos óptimos para cada modelo base
4. Predecir con meta-modelo en test

**Meta-modelo:** Ridge Regression (evita overfitting a un solo modelo base)

In [ ]:
# Generar predicciones de modelos base
def generate_base_predictions(models_dict, X_train, X_test, y_train_col, commodity, model_type):
    """
    Genera predicciones de modelos base
    
    Returns:
        train_preds, test_preds: Arrays con predicciones
    """
    predictions_train = {}
    predictions_test = {}
    
    if model_type == 'baseline':
        # Baseline models requieren scaling
        scaler = models_dict['scaler']
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        for model_name in ['ridge', 'lasso', 'elastic_net']:
            if model_name in models_dict and commodity in models_dict[model_name]:
                model = models_dict[model_name][commodity]
                predictions_train[f'{model_name}'] = model.predict(X_train_scaled)
                predictions_test[f'{model_name}'] = model.predict(X_test_scaled)
    
    elif model_type == 'tree':
        # Tree models no requieren scaling
        for model_name in ['random_forest', 'xgboost', 'lightgbm']:
            if model_name in models_dict and commodity in models_dict[model_name]:
                model = models_dict[model_name][commodity]
                predictions_train[f'{model_name}'] = model.predict(X_train)
                predictions_test[f'{model_name}'] = model.predict(X_test)
    
    return predictions_train, predictions_test

print("✓ Función generate_base_predictions definida")

In [ ]:
# Entrenar stacking ensemble para cada commodity
stacking_models = {}
stacking_results = {}

print(f"\n{'='*80}")
print(f"STACKING ENSEMBLE")
print(f"{'='*80}")

with tqdm(target_cols, desc="Stacking Models", unit="commodity") as pbar:
    for target_col in pbar:
        commodity = target_col.replace('_target_t7', '')
        pbar.set_description(f"Stacking: {commodity}")
        start_time = perf_counter()
        
        print(f"\n--- {commodity} ---")
    
    # 1. Generar predicciones de modelos base
    all_train_preds = {}
    all_test_preds = {}
    
    if baseline_models:
        train_preds, test_preds = generate_base_predictions(
            baseline_models, X_train, X_test, target_col, commodity, 'baseline'
        )
        all_train_preds.update(train_preds)
        all_test_preds.update(test_preds)
    
    if tree_models:
        train_preds, test_preds = generate_base_predictions(
            tree_models, X_train, X_test, target_col, commodity, 'tree'
        )
        all_train_preds.update(train_preds)
        all_test_preds.update(test_preds)
    
    print(f"  Modelos base disponibles: {len(all_train_preds)}")
    print(f"    {', '.join(all_train_preds.keys())}")
    
    # 2. Crear DataFrame con predicciones
    X_train_meta = pd.DataFrame(all_train_preds)
    X_test_meta = pd.DataFrame(all_test_preds)
    
    # 3. Entrenar meta-modelo (Ridge)
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(X_train_meta, y_train[target_col])
    stacking_models[commodity] = {
        'meta_model': meta_model,
        'base_models': list(all_train_preds.keys())
    }
    
    # 4. Predecir
    y_train_pred = meta_model.predict(X_train_meta)
    y_test_pred = meta_model.predict(X_test_meta)
    
    # 5. Evaluar
    train_rmse = np.sqrt(mean_squared_error(y_train[target_col], y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test[target_col], y_test_pred))
    train_r2 = r2_score(y_train[target_col], y_train_pred)
    test_r2 = r2_score(y_test[target_col], y_test_pred)
    
    # Directional accuracy
    y_test_direction = np.sign(y_test[target_col].diff())
    y_test_pred_direction = np.sign(pd.Series(y_test_pred).diff())
    test_dir_acc = (y_test_direction == y_test_pred_direction).mean()
    
    stacking_results[commodity] = {
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'test_dir_acc': test_dir_acc,
        'overfitting_gap': train_r2 - test_r2
    }
    
    # Imprimir resultados
        elapsed = perf_counter() - start_time
        print(f"\n  Resultados Stacking:")
        print(f"    Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
        print(f"    Train R²:   {train_r2:.4f} | Test R²:   {test_r2:.4f}")
        print(f"    Test Dir Acc: {test_dir_acc:.2%}")
        print(f"    Tiempo: {elapsed:.1f}s")
        
        # Pesos del meta-modelo
        print(f"\n  Pesos de modelos base:")
        for model_name, coef in zip(X_train_meta.columns, meta_model.coef_):
            print(f"    {model_name:20s}: {coef:+.4f}")
        
        pbar.set_postfix({'Test_R2': f"{test_r2:.3f}", 'Time': f"{elapsed:.0f}s"})

print(f"\n{'='*80}")

---

## 3. Walk-Forward Validation

**Backtesting realista:** Simula predicción en tiempo real con ventana deslizante.

**Configuración:**
- **Train window:** 252 días (~1 año de trading)
- **Test window:** 21 días (~1 mes de trading)
- **Step:** 21 días (avanza 1 mes cada iteración)

**Proceso:**
1. Entrenar en ventana de 252 días
2. Predecir siguientes 21 días
3. Mover ventana 21 días adelante
4. Repetir hasta agotar datos test

In [ ]:
# Walk-forward validation con mejor modelo por commodity
TRAIN_WINDOW = 252  # 1 año de trading days
TEST_WINDOW = 21    # 1 mes de predicción
STEP = 21           # Avanzar 1 mes

# Seleccionar mejor modelo por commodity (basado en resultados anteriores)
# Por ahora usamos Random Forest como ejemplo
if tree_models and 'random_forest' in tree_models:
    selected_models = tree_models['random_forest']
elif baseline_models and 'ridge' in baseline_models:
    selected_models = baseline_models['ridge']
else:
    print("⚠ No hay modelos disponibles para walk-forward")
    selected_models = None

if selected_models:
    walkforward_results = {}
    
    print(f"\n{'='*80}")
    print(f"WALK-FORWARD VALIDATION")
    print(f"{'='*80}")
    print(f"\nConfiguración:")
    print(f"  Train window: {TRAIN_WINDOW} días")
    print(f"  Test window: {TEST_WINDOW} días")
    print(f"  Step: {STEP} días")
    
    with tqdm(target_cols, desc="Walk-Forward", unit="commodity") as pbar_outer:
        for target_col in pbar_outer:
            commodity = target_col.replace('_target_t7', '')
            pbar_outer.set_description(f"WF: {commodity}")
            start_time = perf_counter()
            
            print(f"\n--- {commodity} ---")
            
            # Preparar datos completos
            X_full = pd.concat([X_train, X_test], axis=0)
            y_full = pd.concat([y_train[target_col], y_test[target_col]], axis=0)
            
            # Walk-forward splits
            n_splits = (len(X_full) - TRAIN_WINDOW) // STEP
            print(f"  N° de splits: {n_splits}")
            
            predictions = []
            actuals = []
            
            for i in tqdm(range(n_splits), desc=f"  {commodity} splits", leave=False):
            # Ventana de entrenamiento
            train_start = i * STEP
            train_end = train_start + TRAIN_WINDOW
            
            # Ventana de test
            test_start = train_end
            test_end = min(test_start + TEST_WINDOW, len(X_full))
            
            if test_end - test_start < TEST_WINDOW // 2:
                break  # No hay suficientes datos para test
            
            # Datos de entrenamiento y test
            X_wf_train = X_full.iloc[train_start:train_end]
            y_wf_train = y_full.iloc[train_start:train_end]
            X_wf_test = X_full.iloc[test_start:test_end]
            y_wf_test = y_full.iloc[test_start:test_end]
            
            # Entrenar modelo
            model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
            model.fit(X_wf_train, y_wf_train)
            
            # Predecir
            y_wf_pred = model.predict(X_wf_test)
            
            # Guardar predicciones y actuals
            predictions.extend(y_wf_pred)
            actuals.extend(y_wf_test)
        
        # Evaluar métricas agregadas
        predictions = np.array(predictions)
        actuals = np.array(actuals)
        
        rmse = np.sqrt(mean_squared_error(actuals, predictions))
        mae = mean_absolute_error(actuals, predictions)
        r2 = r2_score(actuals, predictions)
        
        # Directional accuracy
        direction_actual = np.sign(np.diff(actuals))
        direction_pred = np.sign(np.diff(predictions))
        dir_acc = (direction_actual == direction_pred).mean()
        
            walkforward_results[commodity] = {
                'rmse': rmse,
                'mae': mae,
                'r2': r2,
                'dir_acc': dir_acc,
                'n_predictions': len(predictions)
            }
            
            elapsed = perf_counter() - start_time
            print(f"\n  Resultados Walk-Forward:")
            print(f"    RMSE: {rmse:.4f}")
            print(f"    MAE: {mae:.4f}")
            print(f"    R²: {r2:.4f}")
            print(f"    Directional Acc: {dir_acc:.2%}")
            print(f"    N° predicciones: {len(predictions)}")
            print(f"    Tiempo: {elapsed:.1f}s")
            
            pbar_outer.set_postfix({'Test_R2': f"{r2:.3f}", 'Time': f"{elapsed:.0f}s"})
    
    print(f"\n{'='*80}")

---

## 4. Comparación Final: Todos los Modelos

In [ ]:
# Cargar todos los resultados
baseline_file = PROCESSED_DIR / 'baseline_models_results.json'
tree_file = PROCESSED_DIR / 'tree_models_results.json'
timeseries_file = PROCESSED_DIR / 'time_series_models_results.json'

all_results = []

# Cargar baseline
if baseline_file.exists():
    with open(baseline_file, 'r') as f:
        data = json.load(f)
    all_results.extend(data['results'])

# Cargar tree
if tree_file.exists():
    with open(tree_file, 'r') as f:
        data = json.load(f)
    all_results.extend(data['results'])

# Cargar time series
if timeseries_file.exists():
    with open(timeseries_file, 'r') as f:
        data = json.load(f)
    all_results.extend(data['results'])

# Agregar stacking
for commodity in TARGET_COMMODITIES:
    if commodity in stacking_results:
        r = stacking_results[commodity]
        all_results.append({
            'Commodity': commodity,
            'Model': 'Stacking',
            'Test RMSE': r['test_rmse'],
            'Test MAE': r.get('test_mae', 0),
            'Test R²': r['test_r2'],
            'Test Dir Acc': r['test_dir_acc'],
            'Overfitting Gap': r['overfitting_gap']
        })

# Agregar walk-forward
if 'walkforward_results' in locals():
    for commodity in TARGET_COMMODITIES:
        if commodity in walkforward_results:
            r = walkforward_results[commodity]
            all_results.append({
                'Commodity': commodity,
                'Model': 'Walk-Forward RF',
                'Test RMSE': r['rmse'],
                'Test MAE': r['mae'],
                'Test R²': r['r2'],
                'Test Dir Acc': r['dir_acc'],
                'Overfitting Gap': 0  # No aplica para walk-forward
            })

final_comparison = pd.DataFrame(all_results)

print(f"\n{'='*80}")
print(f"COMPARACIÓN FINAL - TODOS LOS MODELOS")
print(f"{'='*80}\n")

for commodity in TARGET_COMMODITIES:
    print(f"\n--- {commodity} ---")
    commodity_results = final_comparison[final_comparison['Commodity'] == commodity].copy()
    commodity_results = commodity_results.sort_values('Test RMSE')
    
    display(commodity_results[['Model', 'Test RMSE', 'Test R²', 'Test Dir Acc']].head(10))
    
    best_model = commodity_results.iloc[0]['Model']
    best_rmse = commodity_results.iloc[0]['Test RMSE']
    print(f"\n✓ MEJOR MODELO: {best_model} (RMSE: {best_rmse:.4f})")

print(f"\n{'='*80}")

### Visualización: Ranking de Modelos

In [ ]:
# Plot: Top 5 modelos por commodity
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, commodity in enumerate(TARGET_COMMODITIES):
    commodity_results = final_comparison[final_comparison['Commodity'] == commodity].sort_values('Test RMSE')
    top_5 = commodity_results.head(5)
    
    ax = axes[idx]
    
    # Colorear por tipo
    colors = []
    for model in top_5['Model']:
        if 'Stacking' in model or 'Walk-Forward' in model:
            colors.append('gold')
        elif model in ['LSTM', 'Prophet']:
            colors.append('orangered')
        elif model in ['Random Forest', 'XGBoost', 'LightGBM']:
            colors.append('forestgreen')
        else:
            colors.append('steelblue')
    
    ax.barh(range(len(top_5)), top_5['Test RMSE'], color=colors)
    ax.set_yticks(range(len(top_5)))
    ax.set_yticklabels(top_5['Model'], fontsize=10)
    ax.set_xlabel('Test RMSE', fontsize=12)
    ax.set_title(f'{commodity} - Top 5 Models', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='gold', label='Ensemble'),
    Patch(facecolor='orangered', label='Time Series'),
    Patch(facecolor='forestgreen', label='Tree Models'),
    Patch(facecolor='steelblue', label='Baseline')
]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=4, fontsize=11)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(BASE_DIR / 'reports' / 'figures' / 'final_model_ranking.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: reports/figures/final_model_ranking.png")

---

## 5. Guardar Resultados Finales

In [ ]:
# Guardar stacking models
stacking_file = models_dir / 'stacking_models.pkl'
with open(stacking_file, 'wb') as f:
    pickle.dump(stacking_models, f)
print(f"✓ Stacking models guardados: {stacking_file}")

# Guardar resultados finales
final_results = {
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'commodities': TARGET_COMMODITIES,
    'all_results': final_comparison.to_dict(orient='records'),
    'best_models': {},
    'stacking_weights': {},
    'walkforward_config': {
        'train_window': TRAIN_WINDOW,
        'test_window': TEST_WINDOW,
        'step': STEP
    }
}

# Identificar mejor modelo por commodity
for commodity in TARGET_COMMODITIES:
    commodity_results = final_comparison[final_comparison['Commodity'] == commodity].sort_values('Test RMSE')
    final_results['best_models'][commodity] = {
        'model': commodity_results.iloc[0]['Model'],
        'test_rmse': float(commodity_results.iloc[0]['Test RMSE']),
        'test_r2': float(commodity_results.iloc[0]['Test R²']),
        'test_dir_acc': float(commodity_results.iloc[0]['Test Dir Acc'])
    }
    
    # Guardar pesos de stacking
    if commodity in stacking_models:
        meta_model = stacking_models[commodity]['meta_model']
        base_models = stacking_models[commodity]['base_models']
        final_results['stacking_weights'][commodity] = {
            model: float(coef) for model, coef in zip(base_models, meta_model.coef_)
        }

results_file = PROCESSED_DIR / 'final_model_comparison.json'
with open(results_file, 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"✓ Resultados finales guardados: {results_file}")

# Resumen ejecutivo
print(f"\n{'='*80}")
print(f"RESUMEN EJECUTIVO")
print(f"{'='*80}\n")

for commodity in TARGET_COMMODITIES:
    best = final_results['best_models'][commodity]
    print(f"{commodity}:")
    print(f"  Mejor modelo: {best['model']}")
    print(f"  Test RMSE: {best['test_rmse']:.4f}")
    print(f"  Test R²: {best['test_r2']:.4f}")
    print(f"  Directional Acc: {best['test_dir_acc']:.2%}")
    print()

print(f"{'='*80}")